# Data Collection and Feature Extraction with NFStream

**Overview:** Every machine learning project begins with data. In network traffic classification, this means converting the chaotic, high-volume stream of raw network packets into a structured, numerical format suitable for analysis—a process known as **feature extraction**. This chapter demonstrates how to build IP flow datasets using **NFStream**, a high-performance network data analysis framework.

We will follow a step-by-step process of discovery, starting with the most basic flow metering and incrementally adding the features.

**Objective:** To process a PCAP file and generate a labeled, ML-ready dataset.

---

## Setup and Installation

### Project Repository Setup

For this tutorial, we need to ensure all the required files are downloaded.
We start with cloning only the required parts of the `ml-flow-class-tutorial` repository using a sparse checkout. We intentionally avoid downloading the full repo to keep the directory clean.

In [28]:
# Clone the repo but leave the directory empty
!git clone --filter=blob:none --sparse -n https://github.com/FlowFrontiers/ml-flow-class-tutorial.git 2> >(grep -v '^remote:')

# Navigate into the repo and select only the data files we need
!cd ml-flow-class-tutorial && git sparse-checkout set 01-data-collection/pcap 01-data-collection/database 01-data-collection/requirements.txt
!cd ml-flow-class-tutorial && git checkout 2> >(grep -v '^remote:')

# Navigate to the working directory
import os
os.chdir('ml-flow-class-tutorial/01-data-collection')

Cloning into 'ml-flow-class-tutorial'...
Your branch is up to date with 'origin/main'.


After this step, we have:
- the `database/` directory with the files required for GeoIP assignment
- the `pcap/` directory with the required packet traces for this tutorial
- the `requirements.txt` scripts

### The Tool of Choice: NFStream

For this tutorial, our primary tool for data collection and feature extraction is **NFStream**. It is a powerful, high-performance, open-source framework designed specifically for network data analysis. We chose NFStream because it aligns perfectly with the requirements of modern, machine learning-based traffic classification:

-   **Performance:** Its core engine is highly optimized, allowing it to process traffic at high rates, making it suitable for both offline PCAP analysis and live network monitoring.
-   **Rich Feature Set:** It natively calculates a wide array of statistical and temporal features (e.g., packet size distributions, inter-arrival times) that are crucial for classifying traffic without relying on payload inspection, a necessity in an increasingly encrypted internet.
-   **Integrated Labeling:** It incorporates the nDPI library to perform deep packet inspection for application labeling. This provides the ground truth (`application_name`) needed to train and evaluate supervised models.
-   **Extensibility:** As we will demonstrate with advanced examples, NFStream features a powerful plugin system that allows users to inject custom logic for feature enrichment and specialized flow expiration policies.
-   **Python Ecosystem Integration:** It integrates seamlessly with the data science ecosystem, allowing us to directly export data into Pandas DataFrames for immediate use with libraries like Scikit-learn and Plotly.

For a complete list and detailed description of all features generated by NFStream, please refer to our supplementary documentation file, [NFStream guide](https://github.com/FlowFrontiers/ml-flow-class-tutorial/blob/main/01-data-collection/doc/nfstream-guide.md), which we will reference throughout this chapter. For more in-depth information on the library's internal design and the full Application Programming Interface (API), the official documentation is the definitive resource:

-   **Official Design Documentation:** [www.nfstream.org/docs/design](https://www.nfstream.org/docs/design)
-   **Official API Reference:** [www.nfstream.org/docs/api](https://www.nfstream.org/docs/api)

With this understanding of our tool and where to find more information, let's begin our practical exploration.

### NFStream Installation

Before we can start processing network data, we need to install NFStream and some additional packages. The following command will install these.

In [29]:
# Install the required packages
!pip install -r requirements.txt 2>&1 | grep -E "(Instal|Update|Setup|ERROR|FAILED)"

Once the installation is complete, it is a good practice to verify that the library was installed correctly and to check the versions of its key components. We will import NFStream and also check the version of the underlying nDPI library, which handles deep packet inspection.

In [30]:
import nfstream
from nfstream.engine.engine import lib, ffi

# Get the versions of NFStream and its nDPI library
nfstream_version = nfstream.__version__
ndpi_version = ffi.string(lib.engine_lib_ndpi_version()).decode('utf-8')

print(f"NFStream v{nfstream_version} (powered by nDPI v{ndpi_version})")

NFStream v6.5.4 (powered by nDPI v4.7.0-1-4bb8513)


With the versions confirmed, our environment is now ready. Let's begin our practical exploration by examining our first flow.

---

## 1. Flow Metering Fundamentals

### 1.1. The First Encounter: Core Flow Metering

We begin our journey with the simplest possible use case: processing a small packet capture (PCAP) file and inspecting the very first flow NFStream identifies. To focus purely on the foundational L3/L4 flow features, we will explicitly disable Deep Packet Inspection (DPI) for now by setting `n_dissections=0`.

A crucial aspect of any tutorial is **reproducibility**. By default, NFStream may use multiple processing cores (`n_meters=0`) to analyze packets in parallel. While this is excellent for performance, it can lead to slight variations in the order that flows are outputted across different runs. To ensure that our results are identical every time we run this notebook, we will explicitly set `n_meters=1`. This forces single-threaded processing, guaranteeing a deterministic and consistent output order.

Let's create our first `NFStreamer` object and examine a flow.

In [31]:
from nfstream import NFStreamer
import pandas as pd

# Set pandas display options
# pd.set_option('display.max_columns', 500)
# pd.set_option('display.max_rows', 500)
# pd.set_option('display.width', None)  # No width limit - prevents wrapping
# pd.set_option('display.max_colwidth', None)  # No column width limit
# pd.set_option('display.expand_frame_repr', False)  # Don't wrap to multiple lines

# Create a streamer for a small PCAP file.
# n_meters=1 is set for reproducibility.
# n_dissections=0 disables application detection (DPI) to focus on core features.
streamer = NFStreamer(source='pcap/dns_dot.pcap',
                      n_meters=1,
                      n_dissections=0)

print("--- Examining our very first flow object (core features only) ---")
for flow in streamer:
    # Let's just look at the first flow object we get.
    # The 'flow' object is a special NFStream NFlow class instance.
    print(f"Type of object: {type(flow)}\n")

    # Printing the object directly gives a clean, readable summary.
    print("Default flow object representation:")
    print(flow)
    break  # Stop after the first flow.

--- Examining our very first flow object (core features only) ---
Type of object: <class 'nfstream.flow.NFlow'>

Default flow object representation:
NFlow(id=0,
      expiration_id=0,
      src_ip=192.168.1.185,
      src_mac=08:00:27:8d:ab:be,
      src_oui=08:00:27,
      src_port=58290,
      dst_ip=8.8.8.8,
      dst_mac=b8:27:eb:2b:90:f1,
      dst_oui=b8:27:eb,
      dst_port=853,
      protocol=6,
      ip_version=4,
      vlan_id=0,
      tunnel_id=0,
      bidirectional_first_seen_ms=1572783663234,
      bidirectional_last_seen_ms=1572783666246,
      bidirectional_duration_ms=3012,
      bidirectional_packets=24,
      bidirectional_bytes=5869,
      src2dst_first_seen_ms=1572783663234,
      src2dst_last_seen_ms=1572783666246,
      src2dst_duration_ms=3012,
      src2dst_packets=14,
      src2dst_bytes=1480,
      dst2src_first_seen_ms=1572783663269,
      dst2src_last_seen_ms=1572783666246,
      dst2src_duration_ms=2977,
      dst2src_packets=10,
      dst2src_bytes=4389)

#### Analysis of Core Flow Features

With just two simple parameters, NFStream has performed **flow metering**: it has grouped individual packets into a bidirectional flow and extracted a set of core descriptive features. Let's break down what we see:

*   **Flow Identifiers:** We have the complete 5-tuple (`src_ip`, `dst_ip`, `src_port`, `dst_port`, `protocol=6` for TCP) that uniquely defines this flow, along with layer-2 MAC addresses.
*   **Temporal Metrics:** We see the flow's start and end timestamps (`_first_seen_ms`, `_last_seen_ms`) and its total lifetime (`bidirectional_duration_ms=3012`, or ~3 seconds).
*   **Volume Metrics:** We get the total packets and bytes for the entire flow, as well as a directional breakdown (`src2dst` and `dst2src`). Notice the asymmetry: 14 packets sent vs. 10 received, but far more bytes received (4389) than sent (1480), which is typical for a client downloading content from a server.

**Key Takeaway:** This gives us a solid foundation. We have successfully converted raw packets into structured flow records.

---

### 1.2. Working with Flow Data: Output Formats

Now that we understand the basic `NFlow` object, a practical question arises: how do we use this data? A stream of objects is useful for real-time processing, but for analysis, machine learning, or storage, we need to collect the data in structured formats. NFStream provides several convenient methods for this.

#### Method 1: Direct Iteration (Real-time Processing)

This is the method we used above. Iterating directly over the `NFStreamer` object is the most memory-efficient approach, as it processes one flow at a time without storing the entire dataset in memory.

**Use Case:** Ideal for real-time monitoring systems or processing extremely large files that won't fit in RAM.

In [32]:
# We've already seen this, but let's formalize it.
# We'll use a more diverse pcap file for these examples.
streamer_iter = NFStreamer(source='pcap/malware.pcap', n_meters=1, n_dissections=0)

print("--- Processing flows one-by-one via iteration ---")
for i, flow in enumerate(streamer_iter):
    print(f"Flow {i+1}: Protocol={flow.protocol}, Bytes={flow.bidirectional_bytes}")
    if i >= 4: # Stop after 5 flows
        break

--- Processing flows one-by-one via iteration ---
Flow 1: Protocol=17, Bytes=216
Flow 2: Protocol=1, Bytes=98
Flow 3: Protocol=6, Bytes=66
Flow 4: Protocol=6, Bytes=481
Flow 5: Protocol=6, Bytes=7140


#### Method 2: Pandas DataFrame (Data Analysis)

For data science and machine learning, the most common format is a Pandas DataFrame. NFStream makes this incredibly easy with the `.to_pandas()` method.

**Use Case:** The standard choice for data exploration, statistical analysis, visualization, and as the input for machine learning libraries like Scikit-learn.

In [33]:
import pandas as pd

# The .to_pandas() method processes the entire stream and returns a DataFrame.
streamer_pandas = NFStreamer(source='pcap/malware.pcap', n_meters=1, n_dissections=0)
df = streamer_pandas.to_pandas()

print("--- Data loaded into a Pandas DataFrame ---")
print(f"DataFrame shape: {df.shape}")

# Display a subset of relevant columns
# print(df[['id', 'protocol', 'bidirectional_packets', 'bidirectional_bytes']].head())
df.head()

--- Data loaded into a Pandas DataFrame ---
DataFrame shape: (5, 29)


,id,expiration_id,src_ip,src_mac,src_oui,src_port,dst_ip,dst_mac,dst_oui,dst_port,...,src2dst_first_seen_ms,src2dst_last_seen_ms,src2dst_duration_ms,src2dst_packets,src2dst_bytes,dst2src_first_seen_ms,dst2src_last_seen_ms,dst2src_duration_ms,dst2src_packets,dst2src_bytes
0,0,0,192.168.7.7,30:52:cb:6c:9c:1b,30:52:cb,42370,1.1.1.1,08:6a:0a:3a:5e:1e,08:6a:0a,53,...,1569571466977,1569571466977,0,1,106,1569571467001,1569571467001,0,1,110
1,1,0,192.168.7.7,30:52:cb:6c:9c:1b,30:52:cb,0,144.139.247.220,08:6a:0a:3a:5e:1e,08:6a:0a,0,...,1569571470672,1569571470672,0,1,98,0,0,0,0,0
2,2,0,192.168.7.7,30:52:cb:6c:9c:1b,30:52:cb,33706,144.139.247.220,08:6a:0a:3a:5e:1e,08:6a:0a,80,...,1569571476362,1569571476362,0,1,66,0,0,0,0,0
3,3,0,192.168.7.7,30:52:cb:6c:9c:1b,30:52:cb,48394,67.215.92.210,08:6a:0a:3a:5e:1e,08:6a:0a,80,...,1569579408876,1569579408876,0,1,383,1569579409087,1569579409087,0,1,98
4,4,0,192.168.7.7,30:52:cb:6c:9c:1b,30:52:cb,35236,67.215.92.210,08:6a:0a:3a:5e:1e,08:6a:0a,443,...,1569579416636,1569579417280,644,11,1280,1569579416828,1569579417280,452,9,5860


#### Method 3: CSV File (Storage and Sharing)

For long-term storage or for sharing data with colleagues who might use other tools (like R, Excel, or Splunk), exporting to a CSV file is a universal solution.

**Use Case:** Archiving datasets, interoperability with other tools.

In [34]:
import os
os.makedirs('data', exist_ok=True)

# The .to_csv() method processes the stream and writes directly to a file.
streamer_csv = NFStreamer(source='pcap/malware.pcap', n_meters=1, n_dissections=0)
file_path = 'data/flows_output.csv'
num_flows_exported = streamer_csv.to_csv(file_path)

print(f"--- Exported {num_flows_exported} flows to {file_path} ---")

# Let's verify by reading it back with Pandas
df_from_csv = pd.read_csv(file_path)
print("First 5 rows of the CSV file we just created:")
print(df_from_csv.head())

--- Exported 5 flows to data/flows_output.csv ---
First 5 rows of the CSV file we just created:
   id  expiration_id       src_ip            src_mac   src_oui  src_port  \
0   0              0  192.168.7.7  30:52:cb:6c:9c:1b  30:52:cb     42370   
1   1              0  192.168.7.7  30:52:cb:6c:9c:1b  30:52:cb         0   
2   2              0  192.168.7.7  30:52:cb:6c:9c:1b  30:52:cb     33706   
3   3              0  192.168.7.7  30:52:cb:6c:9c:1b  30:52:cb     48394   
4   4              0  192.168.7.7  30:52:cb:6c:9c:1b  30:52:cb     35236   

            dst_ip            dst_mac   dst_oui  dst_port  ...  \
0          1.1.1.1  08:6a:0a:3a:5e:1e  08:6a:0a        53  ...   
1  144.139.247.220  08:6a:0a:3a:5e:1e  08:6a:0a         0  ...   
2  144.139.247.220  08:6a:0a:3a:5e:1e  08:6a:0a        80  ...   
3    67.215.92.210  08:6a:0a:3a:5e:1e  08:6a:0a        80  ...   
4    67.215.92.210  08:6a:0a:3a:5e:1e  08:6a:0a       443  ...   

   src2dst_first_seen_ms  src2dst_last_seen_ms  sr

#### Method 4: Python List (Custom In-Memory Processing)

If you need to perform custom operations that require having all flow objects in memory—for instance, sorting by a specific attribute or running an algorithm that needs multiple passes over the data—you can simply convert the streamer to a list.

**Use Case:** Custom algorithms, sorting, grouping, or any situation where you need random access to all flow objects.

In [35]:
# Convert the streamer iterator directly into a list of NFlow objects
streamer_list = NFStreamer(source='pcap/malware.pcap', n_meters=1, n_dissections=0)
flows_list = list(streamer_list)

print(f"--- Collected {len(flows_list)} flows into a Python list ---")

# Example: Find the top 3 largest flows by byte count
flows_list.sort(key=lambda f: f.bidirectional_bytes, reverse=True)

print("Top 3 largest flows found in the list:")
for i, flow in enumerate(flows_list[:3]):
    print(f"  {i+1}. Src IP: {flow.src_ip}, Bytes: {flow.bidirectional_bytes}")

--- Collected 5 flows into a Python list ---
Top 3 largest flows found in the list:
  1. Src IP: 192.168.7.7, Bytes: 7140
  2. Src IP: 192.168.7.7, Bytes: 481
  3. Src IP: 192.168.7.7, Bytes: 216


#### Takeaways on Output Formats

This exploration demonstrates the flexibility of NFStream's output handling. We've seen that the same underlying flow data can be consumed in multiple ways, each suited for a different purpose.

*   **Direct Iteration** is best for low-memory, real-time streaming applications.
*   **Pandas DataFrames** are the go-to for interactive analysis and machine learning, providing a powerful, tabular view of the data. We can see that the 5 flows from the PCAP have been converted into a DataFrame with 5 rows and 38 columns (the default feature set with DPI enabled).
*   **CSV Export** provides a universal, persistent format for storage and sharing. The resulting file contains all 38 features, ready to be loaded by any data analysis tool.
*   **List Conversion** is a powerful option for custom in-memory algorithms where you need access to the full collection of `NFlow` objects.

For the remainder of this tutorial, we will primarily work with **Pandas DataFrames**, as they are the most convenient format for the data cleaning, feature engineering, and model training tasks that follow.

---

### 1.3. Protecting Privacy: Data Anonymization

Network traffic is inherently sensitive, containing Personally Identifiable Information (PII) like user IP addresses and device MAC addresses. Before sharing or publishing datasets, it is essential to anonymize this information. NFStream provides a built-in, cryptographically secure method for anonymization that replaces sensitive identifiers with irreversible hashes while preserving the analytical integrity of the data.

This is a critical step for two reasons:
1.  **Privacy Compliance:** It helps meet data protection regulations like GDPR.
2.  **Model Generalization:** It forces a machine learning model to learn from behavioral patterns (`_bytes`, `_packets`, `duration`) rather than simply memorizing that "traffic from IP address X is always application Y," which would be a form of data leakage.

Let's see how it works. We will use the `.to_pandas()` method again, but this time we'll pass the `columns_to_anonymize` parameter.

In [36]:
# Re-create the streamer for our pcap file
streamer_anon = NFStreamer(source='pcap/malware.pcap', n_meters=1, n_dissections=0)

# Convert to a DataFrame and specify which columns to anonymize
df_anon = streamer_anon.to_pandas(
    columns_to_anonymize=['src_ip', 'dst_ip', 'src_mac', 'dst_mac']
)

print("--- DataFrame with Anonymized Identifier Columns ---")

# Compare the original vs. anonymized data
# We'll create a simple non-anonymized DataFrame for comparison
df_original = NFStreamer(source='pcap/malware.pcap', n_meters=1).to_pandas()

print("Original src_ip: ", df_original['src_ip'].iloc[0])
print("Anonymized src_ip:", df_anon['src_ip'].iloc[0])
print("\nOriginal dst_mac: ", df_original['dst_mac'].iloc[0])
print("Anonymized dst_mac:", df_anon['dst_mac'].iloc[0])

# Show that analytical relationships are preserved
print("\n--- The anonymized DataFrame is still perfectly usable for analysis ---")
# The 'bidirectional_packets' and 'bidirectional_bytes' columns were not anonymized
print(df_anon[['src_ip', 'bidirectional_packets', 'bidirectional_bytes']].head(3))

--- DataFrame with Anonymized Identifier Columns ---
Original src_ip:  192.168.7.7
Anonymized src_ip: f87649a042fd07a5dec26a9f171a5937f55b4e5f3bdca06a040a5c88d4505e9dc154b285bee8ed19288a7bd902fe3da6cd80e68f48840bd8be3f1a358073c07a

Original dst_mac:  08:6a:0a:3a:5e:1e
Anonymized dst_mac: d8e4015c94d7afa2d7ae1eb3b729b0029ed75957ae22992156d315495cbf03d9e9b4a695aba9a10fab91f001a05f5599cd86d7bb22ac7d570c9abee13e870aa2

--- The anonymized DataFrame is still perfectly usable for analysis ---
                                              src_ip  bidirectional_packets  \
0  f87649a042fd07a5dec26a9f171a5937f55b4e5f3bdca0...                      2   
1  f87649a042fd07a5dec26a9f171a5937f55b4e5f3bdca0...                      1   
2  f87649a042fd07a5dec26a9f171a5937f55b4e5f3bdca0...                      1   

   bidirectional_bytes  
0                  216  
1                   98  
2                   66  


#### Analysis of Anonymization

The output clearly shows the effect of the anonymization process. The original, human-readable `src_ip` (`192.168.7.7`) has been replaced by a long, irreversible cryptographic hash. The same has happened for the `dst_mac` address.

Crucially, all other non-specified columns, like `bidirectional_packets` and `bidirectional_bytes`, remain untouched and fully usable for analysis. This is the key benefit: we have successfully stripped the privacy-sensitive information while preserving the behavioral data that our machine learning models will depend on.

An important property of this hashing is its **consistency within a single export**: the same original IP address will always produce the same anonymized hash during the same `to_pandas()` or `to_csv()` call. This preserves the network topology within each dataset (e.g., you can still count how many flows came from the same, now-anonymized, source). However, **each export uses a fresh random secret key**, so the same IP address will produce different hashes across different exports. This design choice enhances privacy by preventing linkage between separate data exports, while maintaining relational integrity within each individual analysis session.

This concludes our exploration of the basic handling of flow data.

----

## 2. Controlling the Flow Meter: Key Configurations

### 2.1 The Impact of Timeouts on Flow Metering

A network flow is not an infinite entity; it must be "expired" and exported at some point. NFStream, like all flow meters, uses timeouts to determine when a flow is considered complete. Understanding these timeouts is critical, as they directly influence the number and characteristics of the flows you analyze.

NFStream uses two main timeouts:

-   **`idle_timeout` (default: 120s):** A flow is terminated if it sees no traffic for a specified duration. This is common for interactive traffic with pauses (e.g., a user reading a web page before clicking a link).
-   **`active_timeout` (default: 1800s):** A flow is terminated after it has existed for a maximum lifetime, regardless of its activity. This prevents extremely long-lived connections (like a VPN tunnel or a continuous stream) from consuming memory indefinitely.

Let's demonstrate how each timeout affects the metering of a user session captured in `pcap/overleaf.pcap`.

#### Experiment 1: The Effect of `idle_timeout`

We will first process the PCAP with the default 120-second idle timeout. Then, we will process it again with a very aggressive 5-second timeout to see how pauses in the user's interactive session cause the flow to be split.

In [37]:
import pandas as pd
from nfstream import NFStreamer
pd.set_option('display.expand_frame_repr', False)  # Don't wrap to multiple lines

PCAP_PATH = 'pcap/overleaf.pcap'

def run_timeout_experiment(pcap_path, idle_t=120, active_t=1800, description=""):
    """Helper function to run NFStream with specific timeouts and print results."""
    print(f"--- {description} ---")
    streamer = NFStreamer(
        source=pcap_path,
        idle_timeout=idle_t,
        active_timeout=active_t,
        n_meters=1
    )
    df = streamer.to_pandas()
    print(f"Total flows created: {len(df)}")
    if not df.empty:
        # Display key metrics for all resulting flows
        display_cols = ['bidirectional_duration_ms', 'bidirectional_packets', 'bidirectional_bytes', 'bidirectional_first_seen_ms', 'bidirectional_last_seen_ms']
        print(df[display_cols])

# 1. Default idle timeout
run_timeout_experiment(PCAP_PATH, idle_t=120, description="Default Idle Timeout (120s)")

print("\n" + "="*60 + "\n")

# 2. Aggressive idle timeout
run_timeout_experiment(PCAP_PATH, idle_t=5, description="Aggressive Idle Timeout (5s)")

--- Default Idle Timeout (120s) ---
Total flows created: 1
   bidirectional_duration_ms  bidirectional_packets  bidirectional_bytes  bidirectional_first_seen_ms  bidirectional_last_seen_ms
0                      37794                    588               276522                1752647349196               1752647386990


--- Aggressive Idle Timeout (5s) ---
Total flows created: 3
   bidirectional_duration_ms  bidirectional_packets  bidirectional_bytes  bidirectional_first_seen_ms  bidirectional_last_seen_ms
0                       1020                     67                33332                1752647349196               1752647350216
1                       1191                    124                56756                1752647356543               1752647357734
2                      23100                    397               186434                1752647363890               1752647386990


**Analysis of `idle_timeout`:**
-   **Default (120s):** With a generous timeout, the entire 37-second user session is correctly captured as **one single flow**, because no pause in activity was long enough to trigger the timeout.
-   **Aggressive (5s):** The same logical session is now fragmented into **three distinct flows**. This indicates there were at least two periods within the session where the user was idle (e.g., typing or reading) for more than 5 seconds, causing NFStream to cut the flow and start a new one when activity resumed.

#### Experiment 2: The Effect of `active_timeout`

Now, let's demonstrate the `active_timeout`. We will set a very short active timeout of 10 seconds. This will act as a "guillotine," chopping up the flow as soon as its total lifetime exceeds 10 seconds, even if it is continuously active.

In [38]:
# 1. Default active timeout (for reference)
run_timeout_experiment(PCAP_PATH, active_t=1800, description="Default Active Timeout (1800s)")

print("\n" + "="*60 + "\n")

# 2. Aggressive active timeout
run_timeout_experiment(PCAP_PATH, active_t=10, description="Aggressive Active Timeout (10s)")

--- Default Active Timeout (1800s) ---
Total flows created: 1
   bidirectional_duration_ms  bidirectional_packets  bidirectional_bytes  bidirectional_first_seen_ms  bidirectional_last_seen_ms
0                      37794                    588               276522                1752647349196               1752647386990


--- Aggressive Active Timeout (10s) ---
Total flows created: 4
   bidirectional_duration_ms  bidirectional_packets  bidirectional_bytes  bidirectional_first_seen_ms  bidirectional_last_seen_ms
0                       8538                    191                90088                1752647349196               1752647357734
1                       8675                    150                36537                1752647363890               1752647372565
2                       9860                    212               145277                1752647375030               1752647384890
3                       1329                     35                 4620                17526

**Analysis of `active_timeout`:**
-   **Default (1800s):** The flow's total duration of ~38 seconds is well under the 30-minute default, so it is correctly captured as **one flow**.
-   **Aggressive (10s):** The single logical session is now sliced into **four flows**. The first three flows have durations just under the 10-second limit (9.8s, 8.6s, 8.5s). As soon as a flow's lifetime approached 10 seconds, the active timeout triggered, cutting the flow and starting a new one.

**Key Takeaway:** The timeout configuration is a critical parameter that defines the boundaries of a flow. An aggressive `idle_timeout` will fragment interactive sessions with pauses, while an aggressive `active_timeout` will slice long-lived, continuous sessions. The default values are designed for robust, general-purpose analysis, but practitioners must be aware of their impact and adjust them if their specific use case requires a different definition of a flow's lifecycle.

----

### 2.2 Filtering at the Source: Using Berkeley Packet Filters (BPF)

When analyzing large packet captures, you are often interested in only a subset of the traffic—perhaps a specific protocol, host, or network. While you could filter a DataFrame *after* processing, this is highly inefficient. You would waste significant time and memory processing millions of unwanted packets and flows only to discard them later.

A far more efficient method is to filter the traffic at the source. NFStream allows you to do this by passing a **Berkeley Packet Filter (BPF)** string directly to the streamer. This pre-filters packets at the capture level, meaning NFStream's engine never even sees the unwanted traffic.

#### Isolating UDP Traffic

To demonstrate this, we will use `pcap/traffic_trace.pcap`, a file containing a mix of TCP and UDP traffic. We will run a two-part experiment:

1.  **Baseline:** First, we'll process the entire file without a filter to get a baseline count of all protocols present.
2.  **Filtered:** Then, we will re-run the analysis with `bpf_filter="udp"` to prove that only UDP flows are processed.

In [39]:
import pandas as pd
from nfstream import NFStreamer

PCAP_PATH = 'pcap/traffic_trace.pcap'

def analyze_protocol_distribution(df, description=""):
    """Helper function to print the protocol distribution of a DataFrame."""
    print(description)
    if df.empty:
        print("No flows found.")
        return

    # Map protocol numbers to names for readability
    protocol_map = {6: 'TCP', 17: 'UDP'}
    dist = df['protocol'].map(protocol_map).value_counts()
    print(f"Total flows created: {len(df)}")
    print("Protocol distribution:\n", dist)

# --- 1. Process WITHOUT a BPF filter (Baseline) ---
print("--- Processing the entire PCAP without a filter ---")
streamer_full = NFStreamer(source=PCAP_PATH, n_meters=1)
df_full = streamer_full.to_pandas()
analyze_protocol_distribution(df_full, "Baseline Results:")

print("\n" + "="*60 + "\n")

# --- 2. Process WITH a BPF filter ---
print("--- Processing the PCAP with bpf_filter='udp' ---")
streamer_filtered = NFStreamer(
    source=PCAP_PATH,
    bpf_filter="udp", # The BPF filter is applied here
    n_meters=1
)
df_filtered = streamer_filtered.to_pandas()
analyze_protocol_distribution(df_filtered, "Filtered Results:")

--- Processing the entire PCAP without a filter ---
Baseline Results:
Total flows created: 187
Protocol distribution:
 protocol
UDP    109
TCP     75
Name: count, dtype: int64


--- Processing the PCAP with bpf_filter='udp' ---
Filtered Results:
Total flows created: 109
Protocol distribution:
 protocol
UDP    109
Name: count, dtype: int64


#### Analysis of Results

The output provides a clear and unambiguous validation of the BPF filter's effectiveness.

-   **Baseline Run:** Processing the full PCAP file generated **187 total flows**, consisting of 109 UDP flows and 75 TCP flows.
-   **Filtered Run:** After applying `bpf_filter="udp"`, NFStream processed and created exactly **109 flows**. The protocol distribution confirms that every single one of these flows is UDP. The 75 TCP flows were never created because their underlying packets were discarded at the capture stage.

**Key Takeaway:** Using a BPF filter is a fundamental best practice for efficient network analysis. It allows you to focus your computational resources exclusively on the traffic you care about, dramatically reducing processing time and memory consumption, especially when dealing with large-scale packet captures.

----

### 2.3. Ensuring Consistency with Accounting Modes

A fundamental challenge in network analysis is defining a "packet's size." Does it include the Ethernet header? The IP header? Or just the application data? Different tools answer this differently, leading to inconsistent results.

NFStream solves this problem by providing four distinct **accounting modes**. These modes allow you to specify which layer of the network stack you want to measure, ensuring your analysis is consistent and reproducible.

*   **Mode 0 (Default):** Link Layer size (the full packet as seen on the wire).
*   **Mode 1:** IP Layer size (strips link-layer headers, e.g., Ethernet).
*   **Mode 2:** Transport Layer size (strips IP headers, leaving TCP/UDP header + payload).
*   **Mode 3:** Application Payload size (strips all headers, leaving only the application data).

The `splt_ps` feature we just learned about is the perfect tool to visualize the effect of these modes on individual packet sizes. Let's write a script to re-process the same PCAP file four times, once for each mode, and compare the resulting `splt_ps` values for a single flow.

In [40]:
import pandas as pd
from collections import defaultdict

# --- Helper function to process a pcap with a specific mode ---
def process_with_mode(pcap_file, mode):
    streamer = NFStreamer(source=pcap_file,
                          n_meters=1,
                          accounting_mode=mode,
                          splt_analysis=5, # Capture first 5 packets
                          statistical_analysis=False,
                          n_dissections=0)
    # Return the first complex TCP flow we find
    for flow in streamer:
        if flow.protocol == 6 and flow.bidirectional_packets > 4:
            return flow
    return None

# --- Main comparison logic ---
pcap_file = 'pcap/malware.pcap'
mode_results = {}

print("--- Processing PCAP with each accounting mode ---")
for mode_num in range(4):
    print(f"Processing in Mode {mode_num}...")
    mode_results[mode_num] = process_with_mode(pcap_file, mode_num)

# --- Display the comparison table ---
print("\n--- Comparing Packet Sizes (splt_ps) Across Accounting Modes ---")

# Prepare data for the table
table_data = defaultdict(list)
for i in range(5): # First 5 packets
    for mode_num in range(4):
        flow = mode_results[mode_num]
        if flow and i < len(flow.splt_ps):
            table_data[i].append(flow.splt_ps[i])
        else:
            table_data[i].append('-')

# Create a Pandas DataFrame for clean printing
df_comparison = pd.DataFrame.from_dict(table_data, orient='index',
                                       columns=['Mode 0 (Link)', 'Mode 1 (IP)', 'Mode 2 (Transport)', 'Mode 3 (Payload)'])
df_comparison.index.name = 'Packet #'
df_comparison.index = df_comparison.index + 1 # Start packet numbering at 1

print(df_comparison)

--- Processing PCAP with each accounting mode ---
Processing in Mode 0...
Processing in Mode 1...
Processing in Mode 2...
Processing in Mode 3...

--- Comparing Packet Sizes (splt_ps) Across Accounting Modes ---
          Mode 0 (Link)  Mode 1 (IP)  Mode 2 (Transport)  Mode 3 (Payload)
Packet #                                                                  
1                    66           52                  32                 0
2                    66           52                  32                 0
3                    54           40                  20                 0
4                   571          557                 537               517
5                    60           40                  20                 0


#### Analysis of Accounting Modes

The comparison table beautifully illustrates the effect of each accounting mode. Let's analyze the first packet to understand what's happening:

| Mode              | Size (bytes) | Header(s) Stripped                                   | Calculation       |
|:------------------|:------------:|:-----------------------------------------------------|:------------------|
| **Mode 0 (Link)** | 66           | None                                                 | -                 |
| **Mode 1 (IP)**   | 52           | Ethernet Header (14 bytes)                           | 66 - 14 = 52      |
| **Mode 2 (Transport)**| 32           | IPv4 Header (20 bytes)                               | 52 - 20 = 32      |
| **Mode 3 (Payload)**  | 0            | TCP Header (32 bytes with options for this SYN pkt)  | 32 - 32 = 0       |

The results are perfectly consistent:
-   **Mode 0 -> 1:** The size drops by 14 bytes, which is the standard size of an Ethernet II header.
-   **Mode 1 -> 2:** The size drops by 20 bytes, the standard size of an IPv4 header without options.
-   **Mode 2 -> 3:** The size drops by 32 bytes. This is the size of the TCP header for this specific packet (a SYN packet, which often has extra options, making it larger than the base 20 bytes). A key observation is that the payload is 0, which is correct for a TCP SYN packet whose purpose is control, not data transfer.

Let's look at Packet 4, which is carrying application data:
-   **Mode 2 (Transport):** 537 bytes (TCP Header + Payload)
-   **Mode 3 (Payload):** 517 bytes (Payload only)
-   The difference is 20 bytes, the size of a standard TCP header for a data-carrying packet.

**Key Takeaway:** Choosing an accounting mode is a critical decision that affects all byte- and packet-size-based features. The main advantage of NFStream's approach is providing a **consistent abstraction layer** that behaves identically for all supported protocols, including TCP, UDP, ICMP, ICMPv6, over both IPv4 and IPv6.

This consistency gives practitioners great flexibility. For example, you can easily derive precise measurements:
-   **RFC-compliant UDP size:** `mode2_size` (includes UDP header)
-   **Application data only:** `mode3_size`
-   **Transport header size:** `mode2_size - mode3_size`
-   **Total L2-L4 overhead:** `mode0_size - mode3_size`

For our main tutorial, we will standardize on **Mode 2 (Transport)**. This choice strikes a good balance: it removes the variable link-layer and network-layer headers, which can differ between network segments, but retains the transport-layer information, which is often crucial for behavioral analysis.

----

## 3. Generating Features for Machine Learning

### 3.1. Enriching Flows with Statistical Features

<!-- For the purpose of building a sophisticated classifier, simple flow features are insufficient. They describe the "what" (a TCP flow) and the "how much" (volume), but not the detailed "how" (the behavior). We lack the granular, behavioral metrics needed to distinguish this TCP flow from another. -->

So far, we have extracted core flow metrics like duration and total bytes.
While useful, these high-level summaries often lack the detail needed to distinguish between different applications that might have similar session lengths or data volumes.
To build a powerful classifier, especially one that can work with encrypted traffic, we need to analyze the *behavioral fingerprint* of a flow.

This is where statistical analysis becomes essential. By enabling the `statistical_analysis=True` parameter, we instruct NFStream to go beyond simple counters and compute a rich set of statistics on packet sizes and inter-arrival times. These metrics capture the unique rhythm and structure of a communication session.

Let's enable this feature and inspect a flow to see the new information we get.

In [41]:
pd.set_option('display.expand_frame_repr', True)  # Don't wrap to multiple lines

# Enable statistical analysis. For this example, we can keep DPI disabled to focus on the new stats.
streamer_stats = NFStreamer(source='pcap/malware.pcap',
                            n_meters=1,
                            n_dissections=0,
                            statistical_analysis=True)

print("--- Examining a flow with statistical features enabled ---")
for flow in streamer_stats:
    # We'll look for a more complex TCP flow to see a good range of statistics.
    if flow.protocol == 6 and flow.bidirectional_packets > 5:
        print(flow)
        break # Stop after finding one suitable flow.

--- Examining a flow with statistical features enabled ---
NFlow(id=4,
      expiration_id=0,
      src_ip=192.168.7.7,
      src_mac=30:52:cb:6c:9c:1b,
      src_oui=30:52:cb,
      src_port=35236,
      dst_ip=67.215.92.210,
      dst_mac=08:6a:0a:3a:5e:1e,
      dst_oui=08:6a:0a,
      dst_port=443,
      protocol=6,
      ip_version=4,
      vlan_id=0,
      tunnel_id=0,
      bidirectional_first_seen_ms=1569579416636,
      bidirectional_last_seen_ms=1569579417280,
      bidirectional_duration_ms=644,
      bidirectional_packets=20,
      bidirectional_bytes=7140,
      src2dst_first_seen_ms=1569579416636,
      src2dst_last_seen_ms=1569579417280,
      src2dst_duration_ms=644,
      src2dst_packets=11,
      src2dst_bytes=1280,
      dst2src_first_seen_ms=1569579416828,
      dst2src_last_seen_ms=1569579417280,
      dst2src_duration_ms=452,
      dst2src_packets=9,
      dst2src_bytes=5860,
      bidirectional_min_ps=54,
      bidirectional_mean_ps=357.0,
      bidirectional_std

#### Analysis of Statistical Features

The output is now significantly richer. In addition to the core features, we have gained three new categories of behavioral metrics:

1.  **Packet Size (PS) Statistics:**
    -   We now have the `min`, `mean`, `stddev` (standard deviation), and `max` of packet sizes for the bidirectional flow, as well as for each direction individually (`src2dst` and `dst2src`).
    -   For this flow, the `dst2src_max_ps` is 1514 bytes, which is a full-sized packet for a standard Ethernet network (MTU of 1500 + headers). This is a strong indicator of bulk data transfer from the server.

2.  **Packet Inter-Arrival Time (PIAT) Statistics:**
    -   Similarly, we have `min`, `mean`, `stddev`, and `max` for the time elapsed between consecutive packets.
    -   This captures the "rhythm" of the flow. A streaming video might have a very regular PIAT, while interactive web browsing would be much more "bursty" and have a high standard deviation.

3.  **TCP Flag Counts:**
    -   For TCP flows, we now have counters for each type of TCP flag (e.g., `_syn_`, `_ack_`, `_psh_`, `_fin_`).
    -   This provides a detailed fingerprint of the TCP connection's lifecycle. We see `bidirectional_syn_packets=2` (one SYN from client, one SYN-ACK from server) and `bidirectional_fin_packets=2`, indicating a properly established and terminated connection. The `bidirectional_psh_packets=5` suggests data was being actively "pushed" to the application layer, which is common in interactive protocols.

**Key Takeaway:** These statistical features are the lifeblood of modern, ML-based traffic classification. They transform a flow from a simple record of "how much" data was sent into a rich behavioral profile of "how" it was sent. It is these patterns—the size of packets, the timing between them, and the protocol-level handshakes—that will allow our machine learning model to differentiate between applications, even when their content is fully encrypted.

----

### 3.2. Packet-Level Insight with SPLT (Sequence of Packet Lengths and Times)

While statistical summaries (`min`, `mean`, `max`) are powerful, they lose one critical piece of information: **sequence**. The order and pattern of the first few packets in a flow are often a dead giveaway for the application type. For example, a TLS handshake has a very specific back-and-forth sequence of packet sizes.

NFStream allows us to capture this information using **SPLT (Sequence of Packet Lengths and Times)** analysis. By setting `splt_analysis=N`, we instruct NFStream to store the size, direction, and inter-arrival time of the first `N` packets of the flow as three separate lists.

Let's enable SPLT to capture the first 10 packets (`splt_analysis=10`) and see what it reveals about our TCP flow.

In [42]:
# Enable SPLT analysis to capture the first 10 packets.
# We disable statistical analysis for now to focus purely on the new SPLT features.
streamer_splt = NFStreamer(source='pcap/malware.pcap',
                           n_meters=1,
                           n_dissections=0,
                           statistical_analysis=False,
                           splt_analysis=10)

print("--- Examining a flow with SPLT features enabled ---")
for flow in streamer_splt:
    # Find our 20-packet TCP flow again
    if flow.protocol == 6 and flow.bidirectional_packets > 5:
        print(flow)
        break

--- Examining a flow with SPLT features enabled ---
NFlow(id=4,
      expiration_id=0,
      src_ip=192.168.7.7,
      src_mac=30:52:cb:6c:9c:1b,
      src_oui=30:52:cb,
      src_port=35236,
      dst_ip=67.215.92.210,
      dst_mac=08:6a:0a:3a:5e:1e,
      dst_oui=08:6a:0a,
      dst_port=443,
      protocol=6,
      ip_version=4,
      vlan_id=0,
      tunnel_id=0,
      bidirectional_first_seen_ms=1569579416636,
      bidirectional_last_seen_ms=1569579417280,
      bidirectional_duration_ms=644,
      bidirectional_packets=20,
      bidirectional_bytes=7140,
      src2dst_first_seen_ms=1569579416636,
      src2dst_last_seen_ms=1569579417280,
      src2dst_duration_ms=644,
      src2dst_packets=11,
      src2dst_bytes=1280,
      dst2src_first_seen_ms=1569579416828,
      dst2src_last_seen_ms=1569579417280,
      dst2src_duration_ms=452,
      dst2src_packets=9,
      dst2src_bytes=5860,
      splt_direction=[0, 1, 0, 0, 1, 1, 0, 1, 0, 1],
      splt_ps=[66, 66, 54, 571, 60, 1514, 5

#### Analysis of SPLT Features

Enabling SPLT analysis has added three new list-based features to our flow object, telling the story of the first 10 packets:

1.  `splt_direction`: `[0, 1, 0, 0, 1, 1, 0, 1, 0, 1]`
    -   This list shows the direction of each packet: `0` for client-to-server (`src2dst`), `1` for server-to-client (`dst2src`).

2.  `splt_ps` (Packet Sizes): `[66, 66, 54, 571, 60, 1514, 54, 1514, 54, 1514]`
    -   This list gives the size in bytes of each of the first 10 packets.

3.  `splt_piat_ms` (Packet Inter-Arrival Times): `[0, 192, 0, 2, 188, 11, 0, 0, 0, 1]`
    -   This list gives the time in milliseconds that elapsed between each consecutive packet.

#### A Deeper Look at Inter-Arrival Times

Looking at the `splt_piat_ms` list, you might notice several `0` values and wonder if this is an error. It's not; it's an important and expected artifact of how packet timestamps are captured.

The key reason is **timestamp granularity**. NFStreamer calculates these inter-arrival times in **milliseconds (ms)**. When two packets arrive in a rapid burst—separated by only tens or hundreds of *microseconds* (µs)—the time difference is less than 1 ms and is therefore rounded down to `0`.

This bursty behavior is caused by a combination of factors:
*   **Protocol Behavior:** TCP/TLS handshakes are naturally bursty, with acknowledgements sent immediately after receiving data.
*   **System-Level Effects:** Modern systems use techniques like **NIC interrupt coalescing**, where a batch of closely-timed packets is handed to the operating system at once, receiving very similar timestamps.

In contrast, the longer gaps you see (`192` ms and `188` ms) are not artifacts. They represent the true network **Round Trip Time (RTT)**—the time it takes for a packet to travel to the server and for a response to come back. This mix of zero-ms bursts and RTT-scale delays is a normal and highly characteristic feature of network flows.

#### Decoding the Story of the Flow

By combining these three arrays, we can reconstruct the initial conversation of this TCP flow with remarkable detail:

| Pkt | Direction     | Size (bytes) | Time Gap (ms) | Inferred Action                     |
|:---:|---------------|:------------:|:-------------:|-------------------------------------|
| 1   | Client→Server | 66           | 0             | TCP SYN - "Let's connect."          |
| 2   | Server→Client | 66           | 192           | TCP SYN-ACK - "OK, let's connect."  |
| 3   | Client→Server | 54           | 0             | TCP ACK - "Connection established." |
| 4   | Client→Server | 571          | 2             | *Large client packet (e.g., TLS ClientHello)* |
| 5   | Server→Client | 60           | 188           | *Small server ACK*                  |
| 6   | Server→Client | 1514         | 11            | *Large server packet (e.g., TLS ServerHello)* |
| 7   | Client→Server | 54           | 0             | *Small client ACK*                  |
| 8-10| (alternating) | (large/small)| (fast)        | *Data transfer begins*              |

**Key Takeaway:** The SPLT features provide a powerful, granular view of a flow's initial behavior. This sequence data, including the mix of RTT-scale delays and zero-ms bursts, is a highly discriminative fingerprint. While we will focus on statistical summaries for our main classifier, SPLT is an essential tool for deep protocol analysis and can be used to train more advanced sequence-based models (like LSTMs or Transformers).

---


### 3.3. Getting the Ground Truth: Application Labeling

So far, we have built a rich set of behavioral features. We have statistical summaries, packet-level sequences, and a consistent way to measure their sizes. We have everything we need for the *input* to our model (`X`). Now, we need the *output* our model will learn to predict: the **application label** (`y`).

To get this ground truth, we will enable NFStream's Deep Packet Inspection (DPI) engine. NFStream integrates the powerful **nDPI** library, which can identify thousands of applications and protocols by inspecting the content of packet payloads (up to a configured limit).

The key parameter is `n_dissections`. This tells NFStream the maximum number of packets per flow to dissect and send to the nDPI engine for analysis. A value of `n_dissections=20` is a robust choice that allows nDPI to inspect the initial handshake and data exchange of most protocols, leading to high-quality labels.

Let's enable DPI and see what new, label-related features appear.

In [43]:
# Enable DPI by setting n_dissections > 0.
# We'll disable statistical and splt analysis to focus on the new application-layer features.
streamer_labeled = NFStreamer(source='pcap/malware.pcap',
                              n_meters=1,
                              n_dissections=20,
                              statistical_analysis=False,
                              splt_analysis=0)

print("--- Examining a flow with Application Labeling enabled ---")
for flow in streamer_labeled:
    # Find our 20-packet TCP flow again
    if flow.protocol == 6 and flow.bidirectional_packets > 5:
        print(flow)
        break

--- Examining a flow with Application Labeling enabled ---
NFlow(id=4,
      expiration_id=0,
      src_ip=192.168.7.7,
      src_mac=30:52:cb:6c:9c:1b,
      src_oui=30:52:cb,
      src_port=35236,
      dst_ip=67.215.92.210,
      dst_mac=08:6a:0a:3a:5e:1e,
      dst_oui=08:6a:0a,
      dst_port=443,
      protocol=6,
      ip_version=4,
      vlan_id=0,
      tunnel_id=0,
      bidirectional_first_seen_ms=1569579416636,
      bidirectional_last_seen_ms=1569579417280,
      bidirectional_duration_ms=644,
      bidirectional_packets=20,
      bidirectional_bytes=7140,
      src2dst_first_seen_ms=1569579416636,
      src2dst_last_seen_ms=1569579417280,
      src2dst_duration_ms=644,
      src2dst_packets=11,
      src2dst_bytes=1280,
      dst2src_first_seen_ms=1569579416828,
      dst2src_last_seen_ms=1569579417280,
      dst2src_duration_ms=452,
      dst2src_packets=9,
      dst2src_bytes=5860,
      application_name=TLS.OpenDNS,
      application_category_name=Network,
      applic

#### Analysis of Application Layer Features

Success! By enabling DPI, we have added a rich set of application-layer features to our flow object. These are the key to our supervised learning task:

*   `application_name=TLS.OpenDNS`: This is the **fine-grained application label**. nDPI has identified the flow as a TLS session communicating with an OpenDNS server. This will be one of our target variables (`y`).
*   `application_category_name=Network`: This is the **coarse-grained category label**. nDPI groups specific applications into broader categories (e.g., "Web", "SocialNetwork", "Network"). This will be our second target variable.
*   `application_confidence=6`: This indicates the confidence level of the detection. A value of 6 corresponds to `NDPI_CONFIDENCE_DPI`, meaning the classification was made with high confidence based on full deep packet inspection.
*   `requested_server_name=www.internetbadguys.com`: This is the Server Name Indication (SNI) extracted from the TLS handshake, a highly valuable piece of metadata.
*   `client_fingerprint` and `server_fingerprint`: These are JA3/JA3S fingerprints of the TLS handshake. They are powerful features for security analysis but can sometimes be too specific and lead to model overfitting if not handled carefully.

**Key Takeaway:** With application labeling enabled, we have completed the feature set required for our entire project. We have:
1.  **Core Features** (IDs, Timestamps, Volume) from basic metering.
2.  **Behavioral Features** (Statistical and/or SPLT) to describe *how* the flow behaves.
3.  **Ground Truth Labels** (Application Name/Category) to train and evaluate our models.

We are now ready to put all these pieces together.

----

## 4. Advanced Techniques and Real-World Challenges

### 4.1. The Impact of Network Interface Card (NIC) Offloading

A common source of confusion in modern packet analysis is observing packets that appear to be larger than the standard network Maximum Transmission Unit (MTU) of ~1500 bytes. It is not unusual to see "super packets" of up to 64KB in a capture. This is not an error; it is a direct result of performance optimization features in modern NICs and drivers, primarily **Large Receive Offload (LRO)** and its more modern Linux equivalent, **Generic Receive Offloading (GRO)**.

The goal of these features is to reduce CPU overhead. Instead of sending every single small packet up to the CPU for processing, the NIC hardware or its driver reassembles a sequence of related incoming packets into one single, large buffer before passing it to the operating system's network stack.

| Offloading Feature                 | Direction | What it Does                                                                                              |
|:-----------------------------------|:----------|:----------------------------------------------------------------------------------------------------------|
| **TSO** (TCP Segmentation Offload) | Transmit  | The OS sends a large data buffer to the NIC; the NIC hardware segments it into MTU-sized packets for the wire. |
| **GSO** (Generic Segmentation Offload) | Transmit  | A more advanced version of TSO that is aware of different protocols.                                    |
| **LRO/GRO** (Large/Generic Receive Offload) | **Receive**   | **(What we observe)** The NIC receives many MTU-sized packets and reassembles them into a single large buffer before passing it to the OS for capture. |

Packet capture libraries like libpcap (used by NFStream, Wireshark, etc.) often capture traffic *after* this reassembly has occurred. Therefore, the captured "packet" is not what was on the physical wire, but the large, coalesced buffer.

#### The Experiment: Offloading ON vs. OFF

To demonstrate this effect, we will analyze two PCAP files captured from the same home router. For context, key commands exectued on this router were:

```
user@OpenWrt:~# ethtool -k br-lan
Features for br-lan:
...
tcp-segmentation-offload: on
...
generic-segmentation-offload: on
generic-receive-offload: on
large-receive-offload: off [fixed]
...

user@OpenWrt:~# tcpdump -i br-lan -w offloading_on.pcap host 192.168.8.115

user@OpenWrt:~# ethtool -K br-lan tso off gso off gro off
user@OpenWrt:~# tcpdump -i br-lan -w offloading_off.pcap host 192.168.8.115

user@OpenWrt:~# ethtool -K br-lan tso on gso on gro on
```

1.  **`offloading_on.pcap`**: Captured with the default settings, where TSO, GSO, and GRO are enabled.
2.  **`offloading_off.pcap`**: Captured after manually disabling TSO, GSO, and GRO on the router's interface.

We will process both files and compare the `bidirectional_max_ps` and the `splt_ps` of the largest flows found in each.

In [44]:
import pandas as pd
from nfstream import NFStreamer
pd.set_option('display.expand_frame_repr', False)  # Don't wrap to multiple lines

def analyze_max_packet_sizes(pcap_path, description=""):
    """Helper function to find and display the top 3 largest flows by max packet size."""
    print(f"--- {description} ---")
    streamer = NFStreamer(source=pcap_path, n_meters=1, statistical_analysis=True, accounting_mode=1, splt_analysis=15)
    df = streamer.to_pandas()

    if not df.empty:
        # Find the top 5 flows with the largest max packet size
        top_5_largest = df.nlargest(5, 'bidirectional_max_ps')
        print("Top 5 flows by maximum packet size:")
        print(top_5_largest[['bidirectional_max_ps', 'bidirectional_packets', 'bidirectional_bytes', 'splt_ps']])
    else:
        print("No flows found.")

# --- Analyze with NIC Offloading ON ---
PCAP_ON = 'pcap/offloading_on.pcap'
analyze_max_packet_sizes(PCAP_ON, description="Processing with Offloading ENABLED (Default)")

print("\n" + "="*60 + "\n")

# --- Analyze with NIC Offloading OFF ---
PCAP_OFF = 'pcap/offloading_off.pcap'
analyze_max_packet_sizes(PCAP_OFF, description="Processing with Offloading DISABLED")

pd.set_option('display.expand_frame_repr', True)  # Don't wrap to multiple lines

--- Processing with Offloading ENABLED (Default) ---
Top 5 flows by maximum packet size:
    bidirectional_max_ps  bidirectional_packets  bidirectional_bytes                                            splt_ps
11                 64852                    622              2708225  [64, 60, 52, 569, 52, 2645, 1205, 52, 64, 116,...
38                 60028                    386              2182910  [64, 60, 52, 569, 4336, 52, 52, 52, 1541, 52, ...
32                 25972                    132               334015  [64, 60, 52, 569, 52, 1492, 1724, 52, 116, 461...
41                 16446                    152               376103  [64, 52, 40, 557, 40, 2944, 1232, 1250, 1250, ...
45                 15760                     42               106757  [64, 60, 52, 569, 4336, 52, 603, 52, 52, 116, ...


--- Processing with Offloading DISABLED ---
Top 5 flows by maximum packet size:
    bidirectional_max_ps  bidirectional_packets  bidirectional_bytes                                         

#### Analysis of Results

The output provides a stark and unambiguous demonstration of receive offloading in action.

-   **Offloading ENABLED:** The maximum packet sizes (`bidirectional_max_ps`) are enormous, reaching up to **64,852 bytes**. These are the "super packets" created by the router's GRO feature. The `splt_ps` column further reveals packets of varying large sizes like `2645`, `4336`, etc., which are coalesced buffers.

-   **Offloading DISABLED:** The results are completely different. The maximum packet size across all top flows is now capped at **1492 bytes**. This value corresponds to the standard MTU of an Ethernet network, reflecting the actual size of the individual frames that were transmitted over the wire before any reassembly could occur. The `splt_ps` column now clearly shows a sequence of these ~1492-byte packets.

<!-- **Key Takeaway:** It is crucial for a network analyst to be aware of where their data was captured in the network stack. The presence of packets larger than the MTU is a strong indicator that the capture was performed on a host *with receive offloading enabled*. While these super packets accurately reflect what the OS kernel processed, they do **not** represent the individual frames that were actually transmitted over the wire. This understanding is vital for correct performance analysis and for building ML features that are robust to these hardware-level optimizations. -->

#### The Critical Impact on Machine Learning

This is not just a statistical anomaly; it has a profound impact on the performance and generalizability of ML-based classifiers. The two datasets we generated, despite capturing the same logical user activity, have fundamentally different statistical profiles:

-   **Packet Size Distribution:** The range of `max_ps` is `>1492` in one dataset and `>64852` in the other. Features like `mean_ps` and `stddev_ps` will also be drastically different.
-   **Packet Count & Timing:** The offloaded stream has far fewer packets. This means features like `bidirectional_packets` and PIAT statistics will also have completely different distributions.

**Key Takeaway:** A machine learning model trained on data from one environment (e.g., with offloading enabled) will learn a specific set of patterns based on that data's unique statistical distribution. If this model is then deployed in an environment with different offloading settings, it will be presented with data that it has never seen before, leading to a classic **domain shift** problem. The model will likely underperform significantly, not because the model is flawed, but because the underlying data characteristics have changed.

For any practitioner building an ML-based network classifier, this has two critical implications:
1.  **Know Your Data's Provenance:** It is essential to know if our training data was captured with offloading features enabled or disabled.
2.  **Ensure Consistency:** For a model to be reliable, the training environment and the deployment (inference) environment should have **consistent offloading settings**. If this is not possible, the training data *must* be diversified to include samples from both types of environments to have any chance of building a robust and generalizable model.

---

### 4.2. Handling Split Packet Captures

In real-world scenarios, network packet captures are often split into multiple smaller files based on size or time (`tcpdump -C <size>`). This creates a significant challenge for flow analysis known as **boundary flows**: a single, long-lived communication session might start in one PCAP file and end in another.

Processing each file independently is highly problematic. This approach not only fragments the flow's statistics (duration, byte/packet counts) but, more critically, it can lead to a **loss of application context**. The initial packets of a flow, which often contain the protocol handshake (e.g., TLS) and crucial metadata, might only exist in the first PCAP file. Subsequent files would contain only encrypted application data, making it impossible for a DPI engine to identify them correctly.

NFStream elegantly solves this by allowing you to provide a **list of PCAP files** as its source. When given a list, NFStream processes them sequentially but **maintains its internal flow cache** between files, ensuring that boundary flows are correctly stitched together into a single, accurate record with all its metadata intact.

#### Individual vs. Aggregated Processing

To demonstrate this, we will use a set of five PCAP files located in the `pcap/` (`capture-{1..5}.pcap`) directory. While these files contain a mix of traffic, we will use a BPF filter to isolate a single, continuous TLS session to `www.reddit.com` that we know spans across the file boundaries.

We will run the analysis twice and inspect not only the flow statistics but also the application layer metadata:
1.  **Individually:** Processing each of the five PCAPs as a separate stream.
2.  **Aggregated:** Processing all five PCAPs in a single stream.

In [45]:
import os
import pandas as pd
import nfstream

# The BPF filter to isolate the specific TLS session to Reddit
BPF_FILTER = "host 199.232.189.140 and port 52315 and tcp"

def process_pcaps(pcap_directory, bpf_filter, aggregate=False):
    """
    Processes PCAP files from a directory, either individually or as an aggregated list.
    """
    pcap_paths = sorted([
        os.path.join(pcap_directory, f)
        for f in os.listdir(pcap_directory)
        if f.startswith('capture-') and f.endswith('.pcap')
    ])

    if not pcap_paths:
        print(f"No capture-{{1..5}}.pcap files found in {pcap_directory}")
        return

    all_flows_df = pd.DataFrame()

    if aggregate:
        print(f"--- Processing {len(pcap_paths)} PCAPs in AGGREGATED mode ---")
        streamer = nfstream.NFStreamer(
            source=pcap_paths,
            bpf_filter=BPF_FILTER,
            n_dissections=20, # Enable DPI
            n_meters=1
        )
        all_flows_df = streamer.to_pandas()
    else:
        print(f"--- Processing {len(pcap_paths)} PCAPs INDIVIDUALLY ---")
        dfs = []
        for path in pcap_paths:
            streamer = nfstream.NFStreamer(
                source=path,
                bpf_filter=bpf_filter,
                n_dissections=20, # Enable DPI
                n_meters=1
            )
            df = streamer.to_pandas()
            if not df.empty:
                dfs.append(df)
        if dfs:
            all_flows_df = pd.concat(dfs, ignore_index=True)

    print(f"Total flow records created: {len(all_flows_df)}")
    if not all_flows_df.empty:
        # Display key metrics including application metadata
        display_cols = [
            'bidirectional_duration_ms',
            'bidirectional_packets',
            'bidirectional_bytes',
            'requested_server_name',
            'client_fingerprint',
            'server_fingerprint'
        ]
        print(all_flows_df[display_cols])

# --- Run the comparison ---
pcap_dir = "pcap"

# 1. Individual Processing
process_pcaps(pcap_dir, bpf_filter=BPF_FILTER, aggregate=False)

print("\n" + "="*60 + "\n")

# 2. Aggregated Processing
process_pcaps(pcap_dir, bpf_filter=BPF_FILTER, aggregate=True)

--- Processing 5 PCAPs INDIVIDUALLY ---
Total flow records created: 5
   bidirectional_duration_ms  bidirectional_packets  bidirectional_bytes  \
0                       1200                    220               140653   
1                        292                     79                45167   
2                        122                     13                 7720   
3                         84                     13                 4575   
4                         57                      8                 5566   

  requested_server_name                client_fingerprint  \
0        www.reddit.com  773906b0efdefa24a7f2b8eb6985bf37   
1                   NaN                               NaN   
2                   NaN                               NaN   
3                   NaN                               NaN   
4                   NaN                               NaN   

                 server_fingerprint  
0  f4febc55ea12b31ae17cfb7e614afda8  
1                             

#### Analysis of Results

The output provides a definitive and powerful demonstration of why aggregated processing is essential.

-   **Individual Mode (`aggregate=False`):** As predicted, processing each file separately resulted in **5 fragmented flow records**. None of these records represent the true, complete communication. The first record has a duration of 1200 ms, and the last one a mere 57 ms. This is analytically incorrect data. Beyond the corrupted statistics, notice the critical loss of metadata:
    -   Only the *very first fragment* contains the `requested_server_name` (`www.reddit.com`) and the TLS fingerprints.
    -   The subsequent four fragments have `NaN` for these fields. This is because the TLS handshake, which contains this information, occurred only in the first PCAP file. When the other files are processed individually, the DPI engine only sees encrypted application data and has no context to identify the session's metadata. In a real scenario without a BPF filter, these fragments would likely be mislabeled as `Unknown`.

-   **Aggregated Mode (`aggregate=True`):** In stark contrast, providing the list of PCAPs to a single `NFStreamer` instance resulted in **one single, correctly assembled flow record**.
    -   Its `bidirectional_packets` (333) is the sum of the individual fragments (220 + 79 + 13 + 13 + 8 = 333).
    -   Its `bidirectional_bytes` (203,681) is the sum of the fragments' bytes (140,653 + 45,167 + 7,720 + 4,575 + 5,566 = 203,681).
    -   Its `bidirectional_duration_ms` (1909) correctly spans from the very first packet in the first file to the very last packet in the last file.
    -   Most importantly, the vital application layer metadata (`requested_server_name` and fingerprints) is correctly associated with the entire flow, from the first packet to the last.

**Key Takeaway:** This experiment proves that for data integrity, aggregated processing of split captures is non-negotiable. It is essential not only for statistical accuracy but for **preserving critical application-layer context**, which is fundamental for correct protocol identification, security analysis, and high-fidelity traffic classification. When dealing with captures split by time or size, **always provide the file paths as an ordered list to the `NFStreamer` source**.

----


### 4.3. Extending NFStream with Plugins

While NFStream provides a rich set of features out of the box, its true power lies in its extensibility. Through a simple yet powerful plugin system, users can inject custom logic directly into the flow metering process. This allows for a wide range of advanced use cases, including:

-   **Custom Expiration Logic:** Implementing bespoke rules for when a flow should be terminated (e.g., after a specific TCP flag sequence or a certain number of packets).
-   **New Feature Creation:** Enriching flows with new, calculated features that are not natively available (e.g., GeoIP information, ML model inferences, or domain-specific metrics).
-   **Real-time Model Deployment:** Running machine learning models directly on flows as they are being metered for live, on-the-fly classification.

The entire system is built on two core concepts: the `NFPlugin` class, which you inherit from to create your logic, and the `NFPacket` object, which represents the data passed into your plugin.

#### The `NFPlugin` Class Structure

To create a plugin, you must create a class that inherits from [NFPlugin](https://www.nfstream.org/docs/api#nfplugin) and implements one or more of its four main methods. These methods act as "hooks" into the lifecycle of a flow.

<!-- ```python
class NFPlugin(object):
    """ The base NFPlugin class that custom plugins must inherit from. """
    def __init__(self, **kwargs):
        """
        Constructor: Called when the NFStreamer is created.
        Use this to accept and store any configuration your plugin needs.
        """
        pass

    def on_init(self, packet, flow):
        """
        On Flow Creation: Called only for the FIRST packet of a new flow.
        Use this to initialize any custom features on the flow.udps object.
        """
        pass

    def on_update(self, packet, flow):
        """
        On Flow Update: Called for the SECOND and all subsequent packets of an existing flow.
        Use this to update your custom features with new packet information.
        """
        pass

    def on_expire(self, flow):
        """
        On Flow Expiration: Called just before a flow is expired and exported.
        Use this to perform final calculations or decisions based on the completed flow.
        """
        pass

    def cleanup(self):
        """
        On Cleanup: Called when the NFStreamer is being destroyed.
        Use this to release any large resources your plugin might be holding.
        """
        pass
``` -->

#### The `NFPacket` Object

The `on_init` and `on_update` methods are your window into the packet stream. They provide you with an [NFPacket](https://www.nfstream.org/docs/api#nfpacket-object) object, which contains pre-processed information about the packet currently being handled. Its attributes are:

<!-- | Attribute      | Type  | Description                                                 |
|:---------------|:------|:------------------------------------------------------------|
| **Timing**     |       |                                                             |
| `time`         | int   | Packet timestamp in milliseconds.                           |
| `delta_time`   | int   | Time difference in ms since the previous packet in this flow. |
| **Sizing**     |       |                                                             |
| `raw_size`     | int   | Link layer packet size.                                     |
| `ip_size`      | int   | IP layer packet size.                                       |
| `transport_size`| int   | Transport layer segment size.                               |
| `payload_size` | int   | Application layer payload size.                             |
| **Identifiers**|       |                                                             |
| `src_ip`       | str   | Source IP address.                                          |
| `dst_ip`       | str   | Destination IP address.                                     |
| `src_mac`      | str   | Source MAC address.                                         |
| `dst_mac`      | str   | Destination MAC address.                                    |
| `src_port`     | int   | Source transport port.                                      |
| `dst_port`     | int   | Destination transport port.                                 |
| `protocol`     | int   | IP protocol number (e.g., 6 for TCP, 17 for UDP).         |
| **Metadata**   |       |                                                             |
| `ip_version`   | int   | IP version (4 or 6).                                        |
| `direction`    | int   | Packet direction relative to the flow (0: src→dst, 1: dst→src).|
| `vlan_id`      | int   | VLAN identifier (if present).                               |
| `tunnel_id`    | int   | Tunnel identifier (if tunnel decoding is on).               |
| **TCP Flags**  |       |                                                             |
| `syn`, `ack`, `fin`, `rst`, `psh`, `urg`, `ece`, `cwr` | bool | Boolean flags for TCP control bits. |
| **Raw Content**|       |                                                             |
| `ip_packet`    | bytes | Raw packet content starting from the IP header.             | -->


**Key Takeaway:** By implementing methods in an `NFPlugin` subclass, you can inspect each `NFPacket` as it arrives and use its attributes to modify a `flow` object's state. This simple mechanism unlocks a vast potential for customization.

#### 4.3.1. Use Case 1: Custom Expiration with Flow Slicing

Our first practical example of a plugin demonstrates how to implement a custom flow expiration policy. By default, NFStream expires flows based on time (`idle_timeout` or `active_timeout`). However, for certain types of analysis, we might want to "slice" long-lived flows into smaller chunks based on other criteria, such as packet count.

This is useful for creating fixed-size inputs for certain ML models or for analyzing the evolution of a flow's behavior over its lifetime.

##### The `FlowSlicer` Plugin

We will implement a simple `FlowSlicer` plugin. It will take a `limit` (the number of packets) as a parameter. In its `on_update` method, it will check if the flow's total packet count has reached this limit. If it has, it will set the flow's `expiration_id` to `-1`. This is a special value that tells the NFStream engine to force-expire the flow immediately. The packet that triggered the expiration will then become the first packet of a new flow.

In [46]:
from nfstream import NFStreamer, NFPlugin

class FlowSlicer(NFPlugin):
    """
    This plugin expires a flow once it reaches a specified packet count limit.
    """
    def __init__(self, limit=10):
        super().__init__()
        self.limit = limit

    def on_update(self, packet, flow):
        # on_update is called for the 2nd packet onwards
        if flow.bidirectional_packets == self.limit:
           # Setting expiration_id to -1 is the signal to force-expire the flow now.
           flow.expiration_id = -1

##### Before and After Slicing

We will now process a single PCAP file (`pcap/filtered_capture-1.pcap`) that contains one long flow. First, we will process it without the plugin to see the original, complete flow. Then, we will process it again with our `FlowSlicer(limit=10)` to see the effect.

In [47]:
import pandas as pd

# Define the source PCAP and the packet limit for slicing
PCAP_PATH = 'pcap/filtered_capture-1.pcap'
PACKET_LIMIT = 10

# --- 1. Process WITHOUT the slicer plugin ---
print("--- Processing without FlowSlicer (Original Flow) ---")
streamer_original = NFStreamer(source=PCAP_PATH, n_meters=1)
df_original = streamer_original.to_pandas()

print(f"Original flows created: {len(df_original)}")
if not df_original.empty:
    print(df_original[['expiration_id', 'bidirectional_duration_ms', 'bidirectional_packets', 'bidirectional_bytes']])

print("\n" + "="*60 + "\n")

# --- 2. Process WITH the slicer plugin ---
print(f"--- Processing with FlowSlicer(limit={PACKET_LIMIT}) ---")
streamer_sliced = NFStreamer(
    source=PCAP_PATH,
    udps=[FlowSlicer(limit=PACKET_LIMIT)], # Add the plugin here
    n_meters=1
)
df_sliced = streamer_sliced.to_pandas()

print(f"Sliced flows created: {len(df_sliced)}")
if not df_sliced.empty:
    print(df_sliced[['expiration_id', 'bidirectional_duration_ms', 'bidirectional_packets', 'bidirectional_bytes']])

--- Processing without FlowSlicer (Original Flow) ---
Original flows created: 1
   expiration_id  bidirectional_duration_ms  bidirectional_packets  \
0              0                       1200                    220   

   bidirectional_bytes  
0               140653  


--- Processing with FlowSlicer(limit=10) ---
Sliced flows created: 22
    expiration_id  bidirectional_duration_ms  bidirectional_packets  \
0              -1                         49                     10   
1              -1                         29                     10   
2              -1                        140                     10   
3              -1                          1                     10   
4              -1                          4                     10   
5              -1                          0                     10   
6              -1                         10                     10   
7              -1                          3                     10   
8              -1 

##### Analysis of Slicing Results

The output clearly demonstrates the power and precision of the `FlowSlicer` plugin.

-   **Without the Plugin:** As expected, processing the PCAP file normally resulted in **one single flow record**. Its `expiration_id` is `0`, indicating a standard expiration (likely an idle timeout after the PCAP file ended). This record represents the entire 220-packet communication.

-   **With the Plugin:** When we activated the `FlowSlicer(limit=10)`, the result was dramatically different. The single long-lived flow was sliced into **22 distinct flow records**.
    -   The `expiration_id` for every sliced flow is `-1`. This directly confirms that the flow was terminated by our plugin's custom logic (`flow.expiration_id = -1`) and not by a standard timeout.
    -   Each of these flows contains exactly **10 packets**, precisely matching the limit we set. The plugin successfully intercepted the flow update process and forced an expiration each time the packet count hit 10.
    -   The `bidirectional_duration_ms` and `bidirectional_bytes` for each slice are now much smaller, as they only represent the statistics for that 10-packet window.

**Key Takeaway:** The `NFPlugin` system provides a powerful and straightforward mechanism for implementing custom flow expiration logic. By monitoring flow or packet attributes (`on_init`, `on_update`) and setting the `flow.expiration_id` flag, users can gain full control over how flows are defined, metered, and exported, enabling advanced analysis techniques like flow slicing.

---

#### 4.3.2. Use Case 2: Advanced TCP Expiration Policies

By default, NFStream terminates TCP flows based on time, not on TCP's own state-closing signals (the `FIN` and `RST` flags). While this is a robust general-purpose approach, some analyses require more protocol-aware expiration. For example, a security analyst might want to consider a flow "closed" the moment a `RST` (Reset) packet is seen.

The `NFPlugin` system is ideal for implementing such policies. We can inspect the TCP flags on each incoming `NFPacket` and decide whether to force-expire the flow.

##### Defining Two Expiration Policies

We will create two different plugins to demonstrate this:

1.  **`AggressiveExpiry`:** This plugin is very aggressive. It will expire a flow the moment it sees *any* `FIN` or `RST` packet. This is useful for quickly terminating flows that have begun their closing sequence.
2.  **`GracefulExpiry`:** This plugin is more patient. It tries to wait for a "graceful" two-way shutdown. It will only expire the flow after it has seen at least one `FIN` from both sides of the conversation (`bidirectional_fin_packets >= 2`).

In [48]:
from nfstream import NFStreamer, NFPlugin

class AggressiveExpiry(NFPlugin):
    """Expires a flow on the FIRST FIN or RST packet seen."""
    def on_update(self, packet, flow):
        if packet.fin or packet.rst:
            flow.expiration_id = -1 # Force-expire immediately

class GracefulExpiry(NFPlugin):
    """Expires a flow only AFTER a FIN has been seen from both directions."""
    def on_update(self, packet, flow):
        # The bidirectional_fin_packets counter is updated *before* on_update is called.
        # So when the second FIN packet arrives, this counter will be 2.
        if flow.bidirectional_fin_packets >= 2:
            flow.expiration_id = -1

##### Comparing Expiration Behaviors

We will now process a simple PCAP file (`pcap/tcp_stream.pcap`) that contains a single TCP flow with a complete 4-way handshake closure (FIN -> ACK -> FIN -> ACK). We will process it three times:
1.  **Without any plugin** (default time-based expiration).
2.  With the **`AggressiveExpiry`** plugin.
3.  With the **`GracefulExpiry`** plugin.

This will show how the same TCP session is metered into a different number of flows depending on the expiration policy.

In [49]:
import pandas as pd

PCAP_PATH = 'pcap/tcp_stream.pcap'

def run_experiment(pcap_path, plugin=None, description=""):
    print(f"--- {description} ---")
    streamer = NFStreamer(source=pcap_path, udps=plugin, n_meters=1, statistical_analysis=True)
    df = streamer.to_pandas()

    print(f"Total flows created: {len(df)}")
    if not df.empty:
        # Display key metrics that show the flow's completeness
        print(df[['expiration_id', 'bidirectional_duration_ms', 'bidirectional_packets', 'bidirectional_bytes', 'bidirectional_fin_packets', 'bidirectional_ack_packets']])
    return df

# 1. Default Behavior (no plugin)
df_default = run_experiment(PCAP_PATH, description="Default Time-Based Expiration")
print("\n" + "="*70 + "\n")

# 2. Aggressive Expiration
df_aggressive = run_experiment(PCAP_PATH, plugin=[AggressiveExpiry()], description="Aggressive Expiry (on first FIN/RST)")
print("\n" + "="*70 + "\n")

# 3. Graceful Expiration
df_graceful = run_experiment(PCAP_PATH, plugin=[GracefulExpiry()], description="Graceful Expiry (after second FIN)")

--- Default Time-Based Expiration ---
Total flows created: 1
   expiration_id  bidirectional_duration_ms  bidirectional_packets  \
0              0                       2190                   1235   

   bidirectional_bytes  bidirectional_fin_packets  bidirectional_ack_packets  
0              1470989                          2                       1234  


--- Aggressive Expiry (on first FIN/RST) ---
Total flows created: 3
   expiration_id  bidirectional_duration_ms  bidirectional_packets  \
0             -1                       2160                   1231   
1             -1                          1                      3   
2              0                          0                      1   

   bidirectional_bytes  bidirectional_fin_packets  bidirectional_ack_packets  
0              1470725                          1                       1230  
1                  198                          1                          3  
2                   66                          0   

##### Analysis of Expiration Policies

The results clearly show how the same underlying TCP session is metered differently based on the chosen expiration logic.

1.  **Default Behavior:** Without a plugin, NFStream's time-based expiration correctly captures the entire session as **one single flow**. It sees all 1235 packets, including both `FIN` packets, because no timeout was reached during the capture.

2.  **`AggressiveExpiry`:** This policy splits the session into **three flows**.
    -   The first (and largest) flow is expired (`expiration_id=-1`) the moment the *first* `FIN` packet arrives. It contains 1231 packets and only one `FIN` packet.
    -   The second flow begins with the ACK to that first FIN, and then it too is expired when the *second* `FIN` (from the other direction) arrives.
    -   The final, single-packet flow is the last ACK of the 4-way handshake, which creates a new flow that then expires normally.
    This policy gives an immediate view of when a session *begins* to close.

3.  **`GracefulExpiry`:** This more patient policy splits the session into **two flows**.
    -   The main flow continues to accept packets *after* the first `FIN` has been seen. It is only expired (`expiration_id=-1`) when the *second* `FIN` packet arrives. This results in a large flow containing 1234 packets and, crucially, `bidirectional_fin_packets=2`.
    -   The final single-packet flow is again the last ACK of the handshake.
    This policy is useful for identifying sessions that have completed a full, graceful two-way shutdown.

**Key Takeaway:** By inspecting packet-level details like TCP flags within a plugin, you can create highly sophisticated, protocol-aware expiration policies. This allows you to tailor NFStream's definition of a "flow" to match the specific requirements of your analysis, whether it's for security monitoring, performance troubleshooting, or protocol research.

---

#### 4.3.3. Use Case 3: Feature Enrichment with GeoIP
Our final use case moves beyond controlling flow expiration and into the powerful realm of **feature enrichment**. A common task in network analysis is to augment flow records with external data. A prime example is GeoIP lookup, where we translate an IP address into a geographical location (country) and an Autonomous System Number (ASN), which identifies the network operator.

This kind of enrichment adds invaluable context for security analysis (e.g., "Are we seeing traffic from an unusual country?"), business intelligence ("Which network providers are our users on?"), and visualization.

##### The `IPLocateEnrichment` Plugin
We will create a plugin that, upon the creation of a new flow (`on_init`), performs a lookup on the destination IP address. It will use IPLocate's free databases to find the destination's country and ASN information and store these new features on the `flow.udps` object. The `udps` (User-Defined Properties) namespace is the dedicated place to store any custom features you create.

**Prerequisites:** This example requires the following Python libraries:
- `maxminddb` for reading MMDB database files (`pip install maxminddb`)
- `pycountry` for ISO country code conversion (`pip install pycountry`)
- `plotly` for visualization (`pip install plotly`)

You will need to download the free IPLocate databases:
- **ip-to-country.mmdb** - provides country-level geolocation
- **ip-to-asn.mmdb** - provides ASN and organization information

These databases are available from [IPLocate.io](https://github.com/iplocate/ip-address-databases) and should be placed in a `database/` directory in your working directory.

**Attribution:** IP address data powered by [IPLocate.io](https://www.iplocate.com/)

In [50]:
import maxminddb
from nfstream import NFStreamer, NFPlugin
import pandas as pd
import plotly.express as px
import plotly.io as pio
import pycountry
import os

class IPLocateEnrichment(NFPlugin):
    def __init__(self, country_db_path, asn_db_path):
        super().__init__()
        self.country_reader = maxminddb.open_database(country_db_path)
        self.asn_reader = maxminddb.open_database(asn_db_path)

    def on_init(self, packet, flow):
        # Country lookup
        try:
            country_result = self.country_reader.get(flow.dst_ip)
            if country_result:
                flow.udps.dst_country = country_result.get('country_name')
                flow.udps.dst_country_code = country_result.get('country_code')
            else:
                flow.udps.dst_country = None
                flow.udps.dst_country_code = None
        except:
            flow.udps.dst_country = None
            flow.udps.dst_country_code = None

        # ASN lookup
        try:
            asn_result = self.asn_reader.get(flow.dst_ip)
            if asn_result:
                flow.udps.dst_asn = asn_result.get('asn')
                flow.udps.dst_asn_org = asn_result.get('org') or asn_result.get('name')
            else:
                flow.udps.dst_asn = None
                flow.udps.dst_asn_org = None
        except:
            flow.udps.dst_asn = None
            flow.udps.dst_asn_org = None

    def cleanup(self):
        self.country_reader.close()
        self.asn_reader.close()

# Configuration
PCAP_PATH = 'pcap/home_traffic.pcap'
COUNTRY_DB_PATH = 'database/ip-to-country.mmdb'
ASN_DB_PATH = 'database/ip-to-asn.mmdb'

# Check if databases exist
if not (os.path.exists(COUNTRY_DB_PATH) and os.path.exists(ASN_DB_PATH)):
    print(f"ERROR: IPLocate databases not found. Please download them and place them in the database directory.")
else:
    # Process PCAP with IPLocate enrichment
    print("--- Processing with IPLocate Enrichment ---")
    print(f"Using databases:")
    print(f"  Country: {COUNTRY_DB_PATH}")
    print(f"  ASN: {ASN_DB_PATH}")

    streamer = NFStreamer(
        source=PCAP_PATH,
        udps=[IPLocateEnrichment(COUNTRY_DB_PATH, ASN_DB_PATH)],
        n_meters=1
    )

    df = streamer.to_pandas()
    print(f"\nEnriched {len(df)} flows.")

    # Display sample of enriched data
    if not df.empty:
        print("\n--- Sample of IPLocate enriched features ---")
        geo_cols = ['dst_ip', 'udps.dst_country', 'udps.dst_country_code',
                    'udps.dst_asn', 'udps.dst_asn_org']

        # Filter for display columns that actually exist
        display_cols = [col for col in geo_cols if col in df.columns]
        sample_df = df[display_cols].dropna()

        if not sample_df.empty:
            print(sample_df.head(10))

        # Statistics
        df_filtered = df[df['udps.dst_country_code'].notna()]
        print(f"\n--- Geographic Distribution ---")
        print(f"Total flows with country data: {len(df_filtered)} out of {len(df)} ({len(df_filtered)*100/len(df):.1f}%)")

        # Count flows per country
        df_agg = df_filtered['udps.dst_country_code'].value_counts().reset_index()
        df_agg.columns = ['country_code', 'flow_count']

        # Add country names
        country_names = df_filtered.groupby('udps.dst_country_code')['udps.dst_country'].first()
        df_agg['country_name'] = df_agg['country_code'].map(country_names)

        print(f"\nTop 10 destination countries:")
        print(df_agg[['country_name', 'country_code', 'flow_count']].head(10).to_string(index=False))

        # Top ASN organizations
        if 'udps.dst_asn_org' in df.columns:
            print(f"\n--- Top ASN Organizations ---")
            asn_counts = df['udps.dst_asn_org'].value_counts().head(10)
            for org, count in asn_counts.items():
                if org:  # Skip None values
                    print(f"  {org}: {count} flows")

        # Convert ISO-2 to ISO-3 for visualization
        def get_iso3(iso2):
            try:
                return pycountry.countries.get(alpha_2=iso2).alpha_3
            except:
                return iso2

        df_agg['iso3'] = df_agg['country_code'].apply(get_iso3)

        # Create choropleth map
        print("\n--- Generating world map visualization ---")
        fig = px.choropleth(
            df_agg,
            locations='iso3',
            locationmode='ISO-3',
            color='flow_count',
            hover_name='country_name',
            hover_data={'flow_count': ':,', 'country_code': True},
            color_continuous_scale='Reds',
            title="Network Traffic by Country (IPLocate Database)",
            labels={'flow_count': 'Number of Flows'}
        )

        fig.update_layout(
            width=1200,
            height=700,
            geo=dict(
                showframe=False,
                showcoastlines=True,
                projection_type='natural earth'
            )
        )

        fig.show(config={'staticPlot': True})

--- Processing with IPLocate Enrichment ---
Using databases:
  Country: database/ip-to-country.mmdb
  ASN: database/ip-to-asn.mmdb

Enriched 4384 flows.

--- Sample of IPLocate enriched features ---
             dst_ip udps.dst_country udps.dst_country_code  udps.dst_asn  \
0    51.116.253.168          Germany                    DE        8075.0   
2      40.126.53.11           Sweden                    SE        8075.0   
3     52.178.17.235  The Netherlands                    NL        8075.0   
7     20.50.201.204  The Netherlands                    NL        8075.0   
13   17.188.185.134    United States                    US         714.0   
14    40.79.150.120           France                    FR        8075.0   
15  142.250.180.234    United States                    US       15169.0   
18     17.57.146.25    United States                    US         714.0   
21    16.16.163.191           Sweden                    SE       16509.0   
22    20.82.100.209  The Netherlands     

##### Analysis of GeoIP Enrichment
The results powerfully illustrate the value of feature enrichment using IPLocate's free databases.

-   **New Contextual Features:** The `on_init` hook in our plugin successfully executed for each new flow. As the sample DataFrame shows, it added several new columns, such as `udps.dst_asn`, `udps.dst_asn_org`, `udps.dst_country`, and `udps.dst_country_code`, by looking up the destination IP in the IPLocate databases. These user-defined properties are seamlessly integrated into the final output.

-   **Enabling New Visualizations:** The geographical features enable us to move beyond simple tables and create rich, intuitive visualizations. The choropleth map instantly reveals high-level traffic patterns that would be difficult to spot in raw data. The color intensity represents the volume of traffic to each country, making it immediately clear which nations are the primary destinations. In this capture, we can see significant traffic concentrated in the United States, Netherlands, and various European countries, reflecting connections to major cloud providers and content delivery networks.

-   **ASN Intelligence:** The ASN enrichment reveals which network operators and organizations are handling our traffic. This is invaluable for understanding whether connections are going to expected service providers (like Microsoft, Amazon AWS, or Google) or potentially suspicious networks.

**Key Takeaway:** The `NFPlugin` system is a gateway to transforming raw network data into high-level intelligence. By creating enrichment plugins, you can fuse flow data with virtually any external data source—be it geographical information, threat intelligence feeds, or internal asset databases—to create a far richer and more actionable dataset for analysis, visualization, and machine learning. IPLocate's free databases provide an excellent foundation for geographic enrichment without licensing restrictions, making them ideal for open-source projects and educational purposes.

----

## 5. Assembling the Final Dataset

We have systematically investigated NFStream's core capabilities and made principled decisions for each configuration parameter. Now, it is time to combine these choices into a final script to generate the complete, analysis-ready dataset that will fuel the rest of this tutorial.

Here are the choices we are codifying:
-   **Reproducibility:** We will set `n_meters=1` for deterministic output.
-   **Input Features:** We will enable `statistical_analysis=True` to generate the rich behavioral features our model needs.
-   **Target Labels:** We will set `n_dissections=20` to get high-quality application labels from the nDPI engine.
-   **Consistency:** We will set `accounting_mode=2` to standardize our packet size metrics at the transport layer, removing link and network layer variations.
-   **Anonymization:** We will anonymize sensitive identifier columns to protect privacy and promote model generalization.
-   **Output Format:** We will save this dataset in two common formats: the traditional, text-based CSV format and the modern, columnar Parquet format.

The following script encapsulates these decisions and represents the complete data generation process.

In [51]:
import pandas as pd
import numpy as np
import os
from nfstream import NFStreamer

# --- Configuration ---
PCAP_PATH = 'pcap/home_traffic.pcap'
DATA_DIR = 'data'
OUTPUT_CSV_PATH = os.path.join(DATA_DIR, 'dataset.csv')
OUTPUT_PARQUET_PATH = os.path.join(DATA_DIR, 'dataset.parquet')
OUTPUT_PARQUET_OPTIMIZED_PATH = os.path.join(DATA_DIR, 'dataset_optimized_size.parquet')

# Ensure the output directory exists
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Starting dataset generation from: {PCAP_PATH}")

# --- Instantiate the NFStreamer with all our chosen parameters ---
final_streamer = NFStreamer(
    source=PCAP_PATH,
    n_meters=1,
    statistical_analysis=True,
    n_dissections=20,
    accounting_mode=2,
    splt_analysis=0,
    decode_tunnels=True
)

# --- Process the stream and create the final DataFrame with anonymization ---
df_final = final_streamer.to_pandas(
    columns_to_anonymize=['src_ip', 'dst_ip', 'src_mac', 'dst_mac']
)

print(f"Processing complete. Extracted {len(df_final):,} flows.")

# Display a sample of the generated DataFrame
print("\n--- A sample of our generated dataset ---")
display_cols = ['src_ip', 'dst_port', 'protocol', 'bidirectional_duration_ms', 'bidirectional_mean_ps', 'application_category_name', 'application_name']
existing_cols = [col for col in display_cols if col in df_final.columns]
print(df_final[existing_cols].head())

# --- Save to both formats for initial comparison ---
print("\nSaving to CSV and standard Parquet formats...")
df_final.to_csv(OUTPUT_CSV_PATH, index=False)
df_final.to_parquet(OUTPUT_PARQUET_PATH, index=False)
print("Save complete.")

Starting dataset generation from: pcap/home_traffic.pcap
Processing complete. Extracted 4,384 flows.

--- A sample of our generated dataset ---
                                              src_ip  dst_port  protocol  \
0  aa6408d318162ac9dc52d656578a1abb801e75c0807f3a...       443         6   
1  e4e38b958e5074f0ef37ee5c0e2ed25ccb7716e613c402...     49153         6   
2  aa6408d318162ac9dc52d656578a1abb801e75c0807f3a...       443         6   
3  aa6408d318162ac9dc52d656578a1abb801e75c0807f3a...       443         6   
4  c8cdac0c47b23a8aeac37934aba7bf2f4e14883fcbf48b...     49153         6   

   bidirectional_duration_ms  bidirectional_mean_ps application_category_name  \
0                        156             400.357143                       Web   
1                          1              26.000000               Unspecified   
2                        411             780.260870                       Web   
3                        120             525.125000                       W

Now, let's check the size of both CSV file and the standard (non-optimized) Parquet file on disk

In [52]:
import os

# Helper function to get file size in megabytes
def get_file_size_mb(file_path):
    return os.path.getsize(file_path) / (1024 * 1024)

# Get the file sizes
csv_size = get_file_size_mb(OUTPUT_CSV_PATH)
parquet_size = get_file_size_mb(OUTPUT_PARQUET_PATH)

# Calculate the reduction
reduction = (1 - (parquet_size / csv_size)) * 100

print("--- File Size Comparison ---")
print(f"CSV size:      {csv_size:.2f} MB")
print(f"Parquet size:  {parquet_size:.2f} MB")
print(f"Parquet is {reduction:.1f}% smaller than CSV.")

--- File Size Comparison ---
CSV size:      3.83 MB
Parquet size:  0.75 MB
Parquet is 80.4% smaller than CSV.


The initial comparison clearly shows the power of Parquet. With just a few thousand flows, the Parquet file is over 80% smaller than the equivalent CSV. This is because Parquet is a binary, columnar format with highly efficient, built-in compression, whereas CSV is a simple text format. For analytics and machine learning, **Parquet is the superior choice for both storage and query performance.**

We can optimize even further. Pandas often loads numerical data into large default types (like `int64` and `float64`), which use 8 bytes per number. If the values in a column are small enough to fit into a smaller type (like `int32` or `float32`), we can "downcast" them, significantly reducing the DataFrame's memory footprint. This, in turn, will result in an even smaller file when saved.

Let's apply this optimization.

In [53]:
import numpy as np

# Display memory usage before optimization
pre_optimization_memory = df_final.memory_usage(deep=True).sum() / (1024*1024)
print(f"Memory usage before downcasting: {pre_optimization_memory:.2f} MB")

# print("--- Memory Usage Before Downcasting ---")
# df_final.info(memory_usage='deep')

# Downcast integers and floats to the smallest possible type that can hold them
for col in df_final.columns:
    col_type = df_final[col].dtype
    if np.issubdtype(col_type, np.integer):
        df_final[col] = pd.to_numeric(df_final[col], downcast='integer')
    elif np.issubdtype(col_type, np.floating):
        df_final[col] = pd.to_numeric(df_final[col], downcast='float')

post_optimization_memory = df_final.memory_usage(deep=True).sum() / (1024*1024)
print(f"Memory usage after downcasting:  {post_optimization_memory:.2f} MB")

# print("\n--- Memory Usage After Downcasting ---")
# df_final.info(memory_usage='deep')

Memory usage before downcasting: 6.65 MB
Memory usage after downcasting:  5.15 MB


Now, let's save this memory-optimized DataFrame to a new Parquet file and compare all three file sizes.

In [54]:
# --- Save the final, optimized DataFrame ---
df_final.to_parquet(OUTPUT_PARQUET_OPTIMIZED_PATH, index=False)
print("Save complete.")

# --- The Final Comparison ---
optimized_parquet_size = get_file_size_mb(OUTPUT_PARQUET_OPTIMIZED_PATH)
total_reduction = (1 - (optimized_parquet_size / csv_size)) * 100

print("\n--- Final Storage Comparison ---")
print(f"CSV Size:                 {csv_size:.2f} MB")
print(f"Standard Parquet Size:    {parquet_size:.2f} MB")
print(f"Optimized Parquet Size:   {optimized_parquet_size:.2f} MB")
print(f"\nThe final optimized file is {total_reduction:.1f}% smaller than the original CSV.")

Save complete.

--- Final Storage Comparison ---
CSV Size:                 3.83 MB
Standard Parquet Size:    0.75 MB
Optimized Parquet Size:   0.71 MB

The final optimized file is 81.4% smaller than the original CSV.


**Key Takeaway:** By choosing an efficient file format like Parquet and applying data type optimizations like downcasting, we can achieve massive reductions in storage requirements. These practical techniques are essential for creating scalable and cost-effective data pipelines in real-world network analysis projects.

---

## 6. Conclusion

This notebook has provided a comprehensive, end-to-end guide to modern network flow data collection. We have moved systematically from the fundamentals of flow metering to the advanced, real-world challenges that practitioners face.

Through this notebook, we have demonstrated several critical principles:
*   **The importance of configuration:** We proved that choices regarding timeouts, filtering, and accounting modes are not minor details, but fundamental parameters that define the integrity and consistency of the resulting dataset.
*   **The power of rich features:** We showcased the generation of both robust statistical summaries and granular SPLT sequence data, along with the integrated nDPI labeling that provides the essential ground truth for supervised learning.
*   **The necessity of addressing real-world complexities:** Most importantly, we have tackled the non-obvious but critical issues of NIC offloading, split packet captures, and the need for extensibility via plugins.